In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
def padronizar_nomes_rais(nome_mun):
    nome_padronizado =  (nome_mun
                         .replace('-', ' ')
                         .replace(' ', '.')
                        )
    return nome_padronizado

def remover_PI(nome_mun):
    nome_sem_PI = nome_mun.replace(' (PI)', '')
    
    return nome_sem_PI

def tratar_shiftshare_base_sidra(df_shiftshare, ano):
    
    df_shiftshare['NM_MUN_RAIS'] = df_shiftshare['NM_MUN_RAIS'].apply(remover_PI)
    df_shiftshare = df_shiftshare.rename(columns = {'NM_MUN_RAIS': 'NM_MUN'})
    df_shiftshare = df_shiftshare.join(cods_ibge.set_index('NM_MUN'), on = 'NM_MUN')
    df_shiftshare = df_shiftshare[~df_shiftshare['NM_MUN'].isin(['Brasil', 'Norte', 'Sul', 'Sudeste', 'Centro-Oeste','Nordeste', 'Piauí'])]
    df_shiftshare = df_shiftshare[(df_shiftshare['status'] != 2) & (df_shiftshare['ANO_T0'] == ano)]
    #df_shiftshare = df_shiftshare[df_shiftshare['classificacao_regiao'].isin(['T1', 'T2', 'T3', 'T4', 'T5', 'T-1'])]
    
    return df_shiftshare

def formatar_texto_produção(text):

    pattern = r"[X0-9.]"
    filtered_chars = re.findall(pattern, text)

    for char in set(filtered_chars):
        if char == '.':
            text = text.replace(char, ' ')
        else:
            text = text.replace(char, ' ')

    final = ''

    for texto in text.split(' '):
        if texto != '':
            final += texto + ' '
    final = final.rstrip()

    return final

def filtrar(df_shiftshare, mun, tipo):

    df_shiftshare = df_shiftshare[(df_shiftshare['NM_MUN'] == mun) & (df_shiftshare['classificacao_regiao'] == tipo)]

    return df_shiftshare

# Carregar dados 
rais_original = pd.read_csv('./Economia-Regional-R/shit-share-consolidado.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
pam_original = pd.read_csv('./Economia-Regional-R/shift-share-consolidado_pam_prod_agricola.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
pevs_original = pd.read_csv('./Economia-Regional-R/shift-share-consolidado_pevs_prod_extracao_vegetal.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
ppm_aquicultura_original = pd.read_csv('./Economia-Regional-R/shift-share-consolidado_ppm_prod_aquicultura.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
ppm_origem_animal_original = pd.read_csv('./Economia-Regional-R/shift-share-consolidado_ppm_prod_origem_animal.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
ppm_efetivo_rebanho_original = pd.read_csv('./Economia-Regional-R/shift-share-consolidado_ppm_efetivo_rebanhos.csv',
                   decimal = ',',
                   sep = ';',
                   encoding = 'latin1')
# Carregar tabela com nomes e codigos do IBGE para padronização
cods_ibge = pd.read_excel('./Tabelas-Correlacao/cidades-RAIS-IBGE.xlsx')
cods_ibge['NM_MUN_RAIS'] = cods_ibge['NM_MUN_RAIS'].apply(padronizar_nomes_rais)

In [75]:
import pandas as pd
import numpy as np

# =============================================================================
# CORREÇÃO (Passo 1): carregue a base RAIS já filtrada para Brasil.
# O script R corrigido grava 'shift-share-consolidado-Brasil.csv', que contém
# UMA única referência geográfica. Assim cada (município, subclasse) tem uma só
# classificação e as referências Piauí/Nordeste não vazam para o resultado.
#
# Faça o mesmo para as bases do IBGE: elas também devem entrar já restritas à
# comparação 'Brasil'. (Ajuste os caminhos conforme o seu projeto.)
# -----------------------------------------------------------------------------
# rais_original = pd.read_csv('shift-share-consolidado-Brasil.csv',
#                             sep=';', dec=',', encoding='latin1')
# =============================================================================


for ANO in [2013, 2018, 2022]:
    rais = rais_original.join(cods_ibge.set_index('NM_MUN_RAIS'), on='NM_MUN_RAIS')

    # Necessário adicionar o código das subclasses da CNAE para filtrá-las depois
    dicio_caged = pd.read_excel(
        'C:/Users/User/Desktop/Python/Microdados-CAGED-main/Tabelas/dicionário_caged.xlsx',
        sheet_name='subclasse'
    )
    dicio_caged = dicio_caged.rename(columns={'Descrição': 'subclasse'})
    dicio_caged['subclasse'] = dicio_caged['subclasse'].apply(lambda x: x.capitalize())

    # -------------------------------------------------------------------------
    # CORREÇÃO (Passo 2): o join era um-para-muitos.
    # 'subclasse' (descrição com capitalize) NÃO é única no dicionário, então
    # cada linha da RAIS era replicada uma vez por linha repetida do dicionário
    # -> duplicatas exatas. Deduplicamos a chave e usamos merge com validate,
    # que garante 1:1 e levanta erro se a chave ainda não for única.
    # -------------------------------------------------------------------------
    dicio_caged = dicio_caged.drop_duplicates(subset='subclasse', keep='first')
    rais = rais.merge(dicio_caged, on='subclasse', how='left', validate='m:1')

    # Retira agropecuária, comércio e administração pública
    rais_final = rais[((rais['Código'] > 500000) & (rais['Código'] < 4500000)) |
                      ((rais['Código'] > 4900000) & (rais['Código'] < 6300000)) |
                      ((rais['Código'] > 6900000) & (rais['Código'] < 7300000)) |
                      ((rais['Código'] > 8500000) & (rais['Código'] < 9000000))]
    rais_final = rais_final[rais_final['status'] != 2].copy()

    # -------------------------------------------------------------------------
    # CORREÇÃO (Passo 5): a RAIS NÃO estava sendo filtrada nem por referência
    # geográfica nem pelo ano inicial. Por isso cada arquivo recebia TODA a
    # base RAIS (as 3 referências e os 3 anos iniciais empilhados), gerando
    # até 6 classificações para o mesmo setor. Filtramos explicitamente:
    #   (a) só a comparação nacional ('Brasil');
    #   (b) só o ano inicial deste arquivo (ANO).
    # -------------------------------------------------------------------------
    rais_final = rais_final[rais_final['REFERENCIA_GEOGRAFICA'] == 'Brasil']
    rais_final = rais_final[rais_final['ANO_T0'] == ANO]

    rais_final['Fonte'] = np.repeat('RAIS', len(rais_final))

    # Verificação: depois de filtrar, cada (município, subclasse) tem que ter
    # UMA única classificação. Se estourar aqui, ainda há referência/ano vazando.
    _chk = rais_final.groupby(['NM_MUN_RAIS', 'subclasse'])['classificacao_regiao'].nunique()
    assert (_chk <= 1).all(), f"RAIS ainda tem setores com >1 classificação no ano {ANO}"

    pam = tratar_shiftshare_base_sidra(pam_original, ANO)
    pevs = tratar_shiftshare_base_sidra(pevs_original, ANO)
    ppm_aquicultura = tratar_shiftshare_base_sidra(ppm_aquicultura_original, ANO)
    ppm_origem_animal = tratar_shiftshare_base_sidra(ppm_origem_animal_original, ANO)
    ppm_efetivo_rebanho = tratar_shiftshare_base_sidra(ppm_efetivo_rebanho_original, ANO)

    # -------------------------------------------------------------------------
    # CORREÇÃO (Passo 5, continuação): as bases do IBGE também trazem as três
    # referências (Piauí/Nordeste/Brasil). O tratar_... já filtra por ano, mas
    # NÃO pela referência — por isso cada setor ainda aparecia em até 3 tipos.
    # Colapsamos explicitamente para a comparação nacional aqui.
    # -------------------------------------------------------------------------
    bases_ibge = [pam, pevs, ppm_aquicultura, ppm_origem_animal, ppm_efetivo_rebanho]
    bases_ibge = [b[b['REFERENCIA_GEOGRAFICA'] == 'Brasil'].copy() for b in bases_ibge]
    pam, pevs, ppm_aquicultura, ppm_origem_animal, ppm_efetivo_rebanho = bases_ibge

    pam['Fonte'] = np.repeat('PAM', len(pam))
    pevs['Fonte'] = np.repeat('PEVS', len(pevs))
    ppm_aquicultura['Fonte'] = np.repeat('PPM', len(ppm_aquicultura))
    ppm_origem_animal['Fonte'] = np.repeat('PPM', len(ppm_origem_animal))
    ppm_efetivo_rebanho['Fonte'] = np.repeat('PPM', len(ppm_efetivo_rebanho))

    pam['Unidade de medida'] = np.repeat('mil reais', len(pam))
    pevs['Unidade de medida'] = np.repeat('mil reais', len(pevs))
    ppm_aquicultura['Unidade de medida'] = np.repeat('mil reais', len(ppm_aquicultura))
    ppm_origem_animal['Unidade de medida'] = np.repeat('mil reais', len(ppm_origem_animal))
    ppm_efetivo_rebanho['Unidade de medida'] = np.repeat('cabeças', len(ppm_efetivo_rebanho))
    rais_final['Unidade de medida'] = np.repeat('pessoas', len(rais_final))

    for tipo in ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T-1']:

        # ---------------------------------------------------------------------
        # CORREÇÃO (Passo 3): reiniciar o acumulador A CADA tipo.
        # Antes ele era criado uma vez por ANO e nunca limpo, então a aba T2
        # recebia T1+T2, a T3 recebia T1+T2+T3, ... e a T-1 recebia tudo.
        # Reiniciando aqui, cada aba contém apenas o seu próprio tipo.
        # ---------------------------------------------------------------------
        potencialidade_final = pd.DataFrame()

        print(f'Agrupando categorias de produção do tipo {tipo} do ano {ANO}')
        for mun in cods_ibge['NM_MUN'].unique():

            print(f"  Coletando potencialidades {tipo} de {mun}/{ANO}")

            rais_tmp_mun = filtrar(rais_final, mun, tipo)
            pam_tmp_mun = filtrar(pam, mun, tipo)
            pevs_tmp_mun = filtrar(pevs, mun, tipo)
            ppm_aquicultura_tmp_mun = filtrar(ppm_aquicultura, mun, tipo)
            ppm_origem_animal_tmp_mun = filtrar(ppm_origem_animal, mun, tipo)
            ppm_efetivo_rebanho_tmp_mun = filtrar(ppm_efetivo_rebanho, mun, tipo)

            shift_shares = [
                pam_tmp_mun,
                pevs_tmp_mun,
                ppm_aquicultura_tmp_mun,
                ppm_origem_animal_tmp_mun,
                ppm_efetivo_rebanho_tmp_mun,
                rais_tmp_mun
            ]

            for regiao in ['Brasil']:

                for ss in shift_shares:
                    # Salvaguarda: mantém só a comparação nacional. Com a base já
                    # filtrada (Passo 1) isto não remove nada, mas protege caso
                    # alguma base ainda traga outras referências.
                    potencialidades_mun = ss.query("REFERENCIA_GEOGRAFICA == @regiao").copy()
                    if len(potencialidades_mun) != 0:
                        potencialidades_mun['subclasse'] = potencialidades_mun['subclasse'].apply(formatar_texto_produção)
                        potencialidades_salvar = potencialidades_mun#[['subclasse', 'classificacao_regiao', 'NM_MUN', 'Estoque_mun_t1', 'Fonte', 'Unidade de medida']]
                        potencialidade_final = pd.concat([potencialidade_final, potencialidades_salvar])

        # ---------------------------------------------------------------------
        # CORREÇÃO (Passo 4): rede de segurança contra duplicatas exatas que
        # sobrarem (ex.: dois códigos CNAE distintos cujo nome colapsa no mesmo
        # texto após formatar_texto_produção). Remove só linhas 100% idênticas.
        # ---------------------------------------------------------------------
        potencialidade_final = potencialidade_final.drop_duplicates()

        mode = 'w' if tipo == 'T1' else 'a'

        with pd.ExcelWriter(f'Shift_share_todos_tipos_{ANO}_bruto_brasil.xlsx',
                            mode=mode, engine='openpyxl') as writer:
            potencialidade_final.to_excel(writer, sheet_name=f'{tipo}', index=False)

Agrupando categorias de produção do tipo T1 do ano 2013
  Coletando potencialidades T1 de Acauã/2013
  Coletando potencialidades T1 de Agricolândia/2013
  Coletando potencialidades T1 de Água Branca/2013
  Coletando potencialidades T1 de Alagoinha do Piauí/2013
  Coletando potencialidades T1 de Alegrete do Piauí/2013
  Coletando potencialidades T1 de Alto Longá/2013
  Coletando potencialidades T1 de Altos/2013
  Coletando potencialidades T1 de Alvorada do Gurguéia/2013
  Coletando potencialidades T1 de Amarante/2013
  Coletando potencialidades T1 de Angical do Piauí/2013
  Coletando potencialidades T1 de Anísio de Abreu/2013
  Coletando potencialidades T1 de Antônio Almeida/2013
  Coletando potencialidades T1 de Aroazes/2013
  Coletando potencialidades T1 de Aroeiras do Itaim/2013
  Coletando potencialidades T1 de Arraial/2013
  Coletando potencialidades T1 de Assunção do Piauí/2013
  Coletando potencialidades T1 de Avelino Lopes/2013
  Coletando potencialidades T1 de Baixa Grande do R

In [37]:
potencialidade_final

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida
2442879,Aparelhamento de placas e execução de trabalho...,T-1,Acauã,1,RAIS,pessoas
2443116,Construção de edifícios,T-1,Acauã,1,RAIS,pessoas
2443117,Construção de rodovias e ferrovias,T-1,Acauã,10,RAIS,pessoas
2444067,Fabricação de produtos de panificação industrial,T-1,Agricolândia,1,RAIS,pessoas
2444834,Restaurantes e similares,T-1,Agricolândia,5,RAIS,pessoas
...,...,...,...,...,...,...
2744932,Ensino fundamental,T-1,Vera Mendes,125,RAIS,pessoas
2744989,Atividades de apoio à gestão de saúde,T-1,Vera Mendes,38,RAIS,pessoas
2745703,Construção de rodovias e ferrovias,T-1,Vila Nova do Piauí,30,RAIS,pessoas
2746313,Treinamento em desenvolvimento profissional e ...,T-1,Vila Nova do Piauí,2,RAIS,pessoas


In [48]:
somente_t1 = pd.DataFrame()

for ANO in [2013, 2018, 2022]:
    tmp = pd.read_excel(f'Potencialidades_todos_tipos_{ANO}_bruto_brasil.xlsx')
    tmp['ANO'] = np.repeat(ANO, len(tmp))
    
    somente_t1 = pd.concat([somente_t1, tmp])

somente_t1

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO
0,Caprino,T1,Acauã,22014,PPM,cabeças,2013
1,Ovino,T1,Acauã,53442,PPM,cabeças,2013
2,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013
3,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013
4,Lenha,T1,Agricolândia,16,PEVS,mil reais,2013
...,...,...,...,...,...,...,...
1678,Ovino,T1,Vila Nova do Piauí,6722,PPM,cabeças,2022
1679,Galináceos total,T1,Vila Nova do Piauí,9169,PPM,cabeças,2022
1680,Ovino,T1,Wall Ferraz,8016,PPM,cabeças,2022
1681,Galináceos total,T1,Wall Ferraz,19556,PPM,cabeças,2022


In [49]:
somente_t1 = somente_t1.drop_duplicates(subset = ['subclasse', 'NM_MUN'], keep = 'first').sort_values(by = 'NM_MUN')
somente_t1

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO
0,Caprino,T1,Acauã,22014,PPM,cabeças,2013
0,Milho em grão,T1,Acauã,1122,PAM,mil reais,2022
1,Ovos de galinha,T1,Acauã,717,PPM,mil reais,2022
2,Mel de abelha,T1,Acauã,1121,PPM,mil reais,2022
0,Galináceos total,T1,Acauã,21779,PPM,cabeças,2018
...,...,...,...,...,...,...,...
15,"Lanchonetes, casas de chá, de sucos e similares",T1,Água Branca,19,RAIS,pessoas,2013
14,Construção de edifícios,T1,Água Branca,63,RAIS,pessoas,2013
13,Aparelhamento de placas e execução de trabalho...,T1,Água Branca,10,RAIS,pessoas,2013
11,Lenha,T1,Água Branca,14,PEVS,mil reais,2013


In [50]:
def classificar_agropecuaria(subclasse):
    if subclasse in ['Caprino', 'Ovino']:
        return 'Ovinocaprinocultura'
    elif subclasse in ['Galináceos total', 'Ovino', 'Codornas', 'Ovos de galinha',
       'Ovos de codorna']:
        return 'Avicultura'
    elif subclasse in ['Bovino', 'Bubalino', 'Leite']:
        return 'Bovinocultura'
    elif subclasse in ['Equino']:
        return 'Equinocultura'
    elif subclasse in ['Mel de abelha']:
        return 'Apicultura e Meliponicultura'
    elif subclasse in ['Suíno total']:
        return 'Suinocultura'
    elif subclasse in ['Tambacu tambatinga', 'Pintado cachara cachapira e pintachara surubim',
                       'Pirarucu', 'Tilápia', 'Tambaqui', 'Camarão', 'Outros peixes', 'Traíra e trairão',
                      'Piau piapara piauçu piava', 'Carpa', 'Alevinos']:
        return 'Pesca, Piscicultura e Aquicultura'
    elif subclasse in ['Milho em grão', 'Soja em grão', 'Sorgo em grão', 'Batata doce', 'Cebola',
                       'Arroz em casca', 'Fava em grão', 'Arroz em casca', 'Amendoim em casca',
                       'Feijão em grão','Algodão herbáceo em caroço', 'Mandioca', 'Cana de açúcar']:
        return 'Agronegócio'
    elif subclasse in ['Lenha', 'Alimentícios', 'Carvão vegetal', 'Ceras', 'Carnaúba pó',
       'Pequi fruto', 'Madeira em tora', 'Babaçu amêndoa', 'Oleaginosos', 'Maracujá']:
        return 'Extrativismo'
    elif subclasse in ['Uva', 'Melão', 'Tomate', 'Manga', 
                       'Coco da baía', 'Melancia', 'Banana cacho', 'Laranja', 'Castanha de caju']:
        return 'Hortifruticultura'
    else:
        return " "

In [51]:
def classificar_potencialidade(subclasse):
    if subclasse in [
        'Educação infantil - pré-escola', 
        'Educação profissional de nível técnico',
        'Transporte escolar',
        'Educação superior - graduação e pós-graduação',
        'Educação infantil - creche', 
        'Ensino de idiomas', 
        'Ensino médio',
    ]:
        return 'Polo de educação'
    elif subclasse in ['Atividades de fisioterapia']:
        return 'Polo de saúde'
    else:
        return classificar_agropecuaria(subclasse)

In [52]:
dicio_caged = pd.read_excel('C:/Users/matheus.barbosa/Desktop/Python/Microdados-CAGED-main/Tabelas/dicionário_caged.xlsx',
                         sheet_name = 'subclasse')
dicio_caged = dicio_caged.rename(columns = {'Descrição':'subclasse'})
dicio_caged['subclasse'] = dicio_caged['subclasse'].apply(lambda x: x.capitalize())
dicio_caged

,Código,subclasse
0,111301,Cultivo de arroz
1,111302,Cultivo de milho
2,111303,Cultivo de trigo
3,111399,Cultivo de outros cereais não especificados an...
4,112101,Cultivo de algodão herbáceo
...,...,...
1352,9609208,Higiene e embelezamento de animais doméstico
1353,9609299,Outras atividades de serviços pessoais não esp...
1354,9700500,Serviços domésticos
1355,9900800,Organismos internacionais e outras instituiçõe...


In [53]:
rais = somente_t1.query("Fonte == 'RAIS'")
rais_potencialidade = rais.join(dicio_caged.set_index('subclasse'), on = 'subclasse')
rais_potencialidade

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Código
3,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013,4929902
2,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013,2342702
15,Construção de edifícios,T1,Agricolândia,5,RAIS,pessoas,2022,4120400
16,Provedores de acesso às redes de comunicações,T1,Agricolândia,7,RAIS,pessoas,2022,6190601
41,Obras de alvenaria,T1,Alagoinha do Piauí,2,RAIS,pessoas,2022,4399103
...,...,...,...,...,...,...,...,...
17,Atividade médica ambulatorial restrita a consu...,T1,Água Branca,7,RAIS,pessoas,2013,8630503
16,Cartórios,T1,Água Branca,6,RAIS,pessoas,2013,6912500
15,"Lanchonetes, casas de chá, de sucos e similares",T1,Água Branca,19,RAIS,pessoas,2013,5611203
14,Construção de edifícios,T1,Água Branca,63,RAIS,pessoas,2013,4120400


In [54]:
def classificar_potencialidade(subclasse):
    if "Artesanato" in subclasse:
        return "Artesanato"
    else:
        
        codigo = dicio_caged.query(f"subclasse == '{subclasse}'")['Código'].values[0]
        
        if codigo > 500000 and codigo < 1000000:
            return 'Mineração'
        elif codigo > 1000000 and codigo < 4000000:
            return 'Indústria'
        elif codigo > 4100000 and codigo < 4400000:
            return 'Construção'
        elif codigo > 4900000 and codigo < 5400000:
            return 'Transporte, armazenagem e correio'
        elif codigo > 5500000 and codigo < 5700000:
            return 'Alojamento e alimentação'
        elif codigo > 5800000 and codigo < 6400000:
            return 'Informação e comunicação'
        elif codigo > 6900000 and codigo < 7300000:
            return 'Atividades profissionais, científicas e técnicas'
        elif codigo > 8500000 and codigo < 8600000:
            return 'Polo de educação'
        elif codigo > 8600000 and codigo < 9000000:
            return 'Polo de saúde'
        elif codigo > 9000000 and codigo < 9400000:
            return 'Artes, cultura, esporte e recreação'
        elif codigo > 9400000:
            return 'Outras atividades de serviço'


In [55]:
rais_potencialidade

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Código
3,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013,4929902
2,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013,2342702
15,Construção de edifícios,T1,Agricolândia,5,RAIS,pessoas,2022,4120400
16,Provedores de acesso às redes de comunicações,T1,Agricolândia,7,RAIS,pessoas,2022,6190601
41,Obras de alvenaria,T1,Alagoinha do Piauí,2,RAIS,pessoas,2022,4399103
...,...,...,...,...,...,...,...,...
17,Atividade médica ambulatorial restrita a consu...,T1,Água Branca,7,RAIS,pessoas,2013,8630503
16,Cartórios,T1,Água Branca,6,RAIS,pessoas,2013,6912500
15,"Lanchonetes, casas de chá, de sucos e similares",T1,Água Branca,19,RAIS,pessoas,2013,5611203
14,Construção de edifícios,T1,Água Branca,63,RAIS,pessoas,2013,4120400


In [56]:
rais_potencialidade['Potencialidade'] = rais_potencialidade['subclasse'].apply(classificar_potencialidade)

In [57]:
rais_potencialidade

del rais_potencialidade['Código']

rais_potencialidade

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade
3,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013,"Transporte, armazenagem e correio"
2,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013,Indústria
15,Construção de edifícios,T1,Agricolândia,5,RAIS,pessoas,2022,Construção
16,Provedores de acesso às redes de comunicações,T1,Agricolândia,7,RAIS,pessoas,2022,Informação e comunicação
41,Obras de alvenaria,T1,Alagoinha do Piauí,2,RAIS,pessoas,2022,Construção
...,...,...,...,...,...,...,...,...
17,Atividade médica ambulatorial restrita a consu...,T1,Água Branca,7,RAIS,pessoas,2013,Polo de saúde
16,Cartórios,T1,Água Branca,6,RAIS,pessoas,2013,"Atividades profissionais, científicas e técnicas"
15,"Lanchonetes, casas de chá, de sucos e similares",T1,Água Branca,19,RAIS,pessoas,2013,Alojamento e alimentação
14,Construção de edifícios,T1,Água Branca,63,RAIS,pessoas,2013,Construção


In [58]:
agro = somente_t1.query("Fonte != 'RAIS'")
agro['Potencialidade'] = agro['subclasse'].apply(classificar_agropecuaria)
agro

C:\Users\matheus.barbosa\AppData\Local\Temp\ipykernel_34716\105050162.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agro['Potencialidade'] = agro['subclasse'].apply(classificar_agropecuaria)


,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade
0,Caprino,T1,Acauã,22014,PPM,cabeças,2013,Ovinocaprinocultura
0,Milho em grão,T1,Acauã,1122,PAM,mil reais,2022,Agronegócio
1,Ovos de galinha,T1,Acauã,717,PPM,mil reais,2022,Avicultura
2,Mel de abelha,T1,Acauã,1121,PPM,mil reais,2022,Apicultura e Meliponicultura
0,Galináceos total,T1,Acauã,21779,PPM,cabeças,2018,Avicultura
...,...,...,...,...,...,...,...,...
19,Melancia,T1,Água Branca,378,PAM,mil reais,2022,Hortifruticultura
21,Oleaginosos,T1,Água Branca,2,PEVS,mil reais,2022,Extrativismo
22,Babaçu amêndoa,T1,Água Branca,2,PEVS,mil reais,2022,Extrativismo
11,Lenha,T1,Água Branca,14,PEVS,mil reais,2013,Extrativismo


In [59]:
pam

,NE,IM,CE,RIE,RSE,RCCE,Estoque_mun_t0,Estoque_mun_t1,Estoque_nac_t0,Estoque_nac_t1,...,subclasse,NM_MUN,ANO_T0,ANO_T1,REFERENCIA_GEOGRAFICA,CD_MUN,NM_MUN_RAIS,CD_MUN_6DIG,Fonte,Unidade de medida
129183,-38.659731,21.604640,171.055091,-115.994627,308.654357,-192.659731,1036,1190,12375867,12172130,...,Feijão..em.grão.,Acauã,2022,2024,Brasil,2200053.0,PI.ACAUA,220005.0,PAM,mil reais
129197,-1.119490,5.936444,-0.816954,-3.818377,8.937867,-5.119490,30,34,15617793,18125466,...,Mandioca,Acauã,2022,2024,Brasil,2200053.0,PI.ACAUA,220005.0,PAM,mil reais
129203,-29.666492,-257.663318,614.329810,119.813004,236.853488,-356.666492,795,1122,137990922,88118085,...,Milho..em.grão.,Acauã,2022,2024,Brasil,2200053.0,PI.ACAUA,220005.0,PAM,mil reais
129228,-6.082564,77.365563,-60.282999,9.271025,7.811539,-17.082564,163,174,15527125,22317432,...,Arroz..em.casca.,Agricolândia,2022,2024,Brasil,2200103.0,PI.AGRICOLANDIA,220010.0,PAM,mil reais
129244,-1.679235,9.302817,13.376419,20.522676,2.156560,-22.679235,45,66,589471,689335,...,Castanha.de.caju,Agricolândia,2022,2024,Brasil,2200103.0,PI.AGRICOLANDIA,220010.0,PAM,mil reais
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144990,-0.485112,6.170260,-16.685147,-8.168814,-2.346073,10.514888,13,2,15527125,22317432,...,Arroz..em.casca.,Wall Ferraz,2022,2024,Brasil,2211704.0,PI.WALL.FERRAZ,221170.0,PAM,mil reais
145006,-1.641919,9.096087,-25.454168,-8.417526,-7.940555,16.358081,44,26,589471,689335,...,Castanha.de.caju,Wall Ferraz,2022,2024,Brasil,2211704.0,PI.WALL.FERRAZ,221170.0,PAM,mil reais
145016,-37.876087,21.166708,-372.290621,-167.949742,-183.174170,351.123913,1015,626,12375867,12172130,...,Feijão..em.grão.,Wall Ferraz,2022,2024,Brasil,2211704.0,PI.WALL.FERRAZ,221170.0,PAM,mil reais
145030,-7.836432,41.555110,162.281322,241.734536,-37.898104,-203.836432,210,406,15617793,18125466,...,Mandioca,Wall Ferraz,2022,2024,Brasil,2211704.0,PI.WALL.FERRAZ,221170.0,PAM,mil reais


In [60]:
final = pd.concat([rais_potencialidade, agro]).reset_index(drop = True)

In [61]:
final[final['Potencialidade'] == ' ']['subclasse'].value_counts().index

Index(['Outros'], dtype='object', name='subclasse')

In [36]:
with pd.ExcelWriter(f'somente_t1_categorizado_2025.xlsx', engine='openpyxl') as writer:  
    final.to_excel(writer)      

In [73]:
final[(final['Fonte'] == 'PPM')&(final['NM_MUN'] == 'Acauã')]#['classificacao_regiao'].value_counts()

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade
1131,Caprino,T1,Acauã,22014,PPM,cabeças,2013,Ovinocaprinocultura
1133,Ovos de galinha,T1,Acauã,717,PPM,mil reais,2022,Avicultura
1134,Mel de abelha,T1,Acauã,1121,PPM,mil reais,2022,Apicultura e Meliponicultura
1135,Galináceos total,T1,Acauã,21779,PPM,cabeças,2018,Avicultura
1136,Ovino,T1,Acauã,53442,PPM,cabeças,2013,Ovinocaprinocultura


In [74]:
tmp = pd.read_csv('C:/Users/matheus.barbosa/Downloads/potencialidades_consolidado_v2.csv', sep = ';')
tmp[(tmp['Fonte'] == 'PPM')&(tmp['NM_MUN'] == 'Acauã')&(tmp['classificacao_regiao'] == 'T1')]#['classificacao_regiao'].value_counts()

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_ano_final,Unidade_de_medida,Fonte,ano_inicial,ano_final
0,Caprino,T1,Acauã,22014,cabeças,PPM,2013,2024
1,Ovino,T1,Acauã,53442,cabeças,PPM,2013,2024
17222,Galináceos total,T1,Acauã,21779,cabeças,PPM,2018,2024
34343,Ovos de galinha,T1,Acauã,717,mil reais,PPM,2022,2024
34344,Mel de abelha,T1,Acauã,1121,mil reais,PPM,2022,2024
34345,Ovino,T1,Acauã,53442,cabeças,PPM,2022,2024
34346,Galináceos total,T1,Acauã,21779,cabeças,PPM,2022,2024


In [181]:
pd.read_excel('C:/Users/matheus.barbosa/Desktop/Repositorios/Indicadores-Tabelas-Essenciais/Tabelas-Correlacao/territorios_desenvolvimento.xlsx')

,CD_MUN,NM_MUN,pi_micro_m,CD_MUN_6DIG
0,2200053,Acauã,Chapada Vale do Itaim,220005
1,2200103,Agricolândia,Entre Rios,220010
2,2200202,Água Branca,Entre Rios,220020
3,2200251,Alagoinha do Piauí,Vale do Rio Guaribas,220025
4,2200277,Alegrete do Piauí,Vale do Rio Guaribas,220027
...,...,...,...,...
219,2211357,Várzea Branca,Serra da Capivara,221135
220,2211407,Várzea Grande,Vale do Sambito,221140
221,2211506,Vera Mendes,Vale do Rio Guaribas,221150
222,2211605,Vila Nova do Piauí,Vale do Rio Guaribas,221160


In [182]:
formatado = pd.pivot_table(final, index = 'NM_MUN', columns =  'Potencialidade', values = 'subclasse',
               aggfunc = 'count', fill_value=0)
formatado

Potencialidade,,Agronegócio,Alojamento e alimentação,Apicultura e Meliponicultura,"Atividades profissionais, científicas e técnicas",Avicultura,Bovinocultura,Construção,Equinocultura,Extrativismo,Hortifruticultura,Indústria,Informação e comunicação,Mineração,Ovinocaprinocultura,"Pesca, Piscicultura e Aquicultura",Polo de educação,Polo de saúde,Suinocultura,"Transporte, armazenagem e correio"
NM_MUN,,,,,,,,,,,,,,,,,,,,
Acauã,0,1,0,1,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,1
Agricolândia,0,2,0,0,0,1,0,1,1,2,1,0,1,0,1,0,0,0,1,0
Alagoinha do Piauí,0,3,0,1,0,0,0,2,0,1,1,0,0,0,0,0,0,0,0,0
Alegrete do Piauí,0,1,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Alto Longá,0,3,0,0,1,1,1,1,0,0,1,2,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vila Nova do Piauí,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
Várzea Branca,0,2,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
Várzea Grande,0,2,0,0,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0


In [183]:
with pd.ExcelWriter(f'potencialidades_com_numeros_2025.xlsx', engine='openpyxl') as writer:  
    formatado.to_excel(writer, index = False)  

In [184]:
#del formatado[' ']
def trocar(x):
    valor = x
    if valor != 0:
        valor = 1
    return valor
for c in formatado.columns:
    formatado[c] = formatado[c].apply(trocar)

formatado = cods_ibge.set_index('NM_MUN')[['CD_MUN']].join(formatado, on = 'NM_MUN').reset_index()
formatado

,NM_MUN,CD_MUN,,Agronegócio,Alojamento e alimentação,Apicultura e Meliponicultura,"Atividades profissionais, científicas e técnicas",Avicultura,Bovinocultura,Construção,...,Hortifruticultura,Indústria,Informação e comunicação,Mineração,Ovinocaprinocultura,"Pesca, Piscicultura e Aquicultura",Polo de educação,Polo de saúde,Suinocultura,"Transporte, armazenagem e correio"
0,Acauã,2200053,0,1,0,1,0,1,0,0,...,0,1,0,0,0,0,0,0,0,1
1,Agricolândia,2200103,0,1,0,0,0,1,0,1,...,1,0,1,0,1,0,0,0,1,0
2,Água Branca,2200202,0,1,1,0,1,1,0,1,...,1,1,1,0,1,0,1,1,1,0
3,Alagoinha do Piauí,2200251,0,1,0,1,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
4,Alegrete do Piauí,2200277,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Várzea Branca,2211357,0,1,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
220,Várzea Grande,2211407,0,1,0,0,1,1,0,0,...,0,0,0,0,1,1,0,0,0,0
221,Vera Mendes,2211506,0,1,0,1,0,1,0,0,...,0,1,0,0,1,0,0,0,0,0
222,Vila Nova do Piauí,2211605,0,1,0,1,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0


In [185]:
with pd.ExcelWriter(f'potencialidades_binario_novo.xlsx', engine='openpyxl') as writer:  
    formatado.to_excel(writer, index = False)  

In [186]:
antigo= pd.read_excel('potencialidades_binario.xlsx')
novo = pd.read_excel('potencialidades_binario_novo.xlsx')

In [187]:
pot = 'Indústria'

boi = [m for m in antigo[antigo[pot] > 0]['NM_MUN'] if m not in novo[novo[pot] > 0]['NM_MUN']]
mel = [m for m in antigo[antigo['Apicultura e Meliponicultura'] > 0]['NM_MUN'] if m not in novo[novo['Apicultura e Meliponicultura'] > 0]['NM_MUN']]

In [188]:
pd.DataFrame(boi).to_csv('boi_tirar.csv', encoding ='latin1')

In [189]:
t1 = pd.read_excel('somente_t1_categorizado_2025.xlsx', index_col = 0)
terr = pd.read_excel('C:/Users/matheus.barbosa/Desktop/Repositorios/Indicadores-Tabelas-Essenciais/Tabelas-Correlacao/territorios_desenvolvimento.xlsx')

terr_t1 = t1.merge(terr[['NM_MUN', 'pi_micro_m']], on = 'NM_MUN')[['pi_micro_m', 'subclasse', 'Potencialidade', 'Estoque_mun_t1']]

terr_t1 

,pi_micro_m,subclasse,Potencialidade,Estoque_mun_t1
0,Chapada Vale do Itaim,Fabricação de artefatos de cerâmica e barro co...,Indústria,11
1,Chapada Vale do Itaim,"Transporte rodoviário coletivo de passageiros,...","Transporte, armazenagem e correio",8
2,Entre Rios,Provedores de acesso às redes de comunicações,Informação e comunicação,7
3,Entre Rios,Construção de edifícios,Construção,5
4,Vale do Rio Guaribas,Obras de alvenaria,Construção,2
...,...,...,...,...
2404,Entre Rios,Ovino,Ovinocaprinocultura,148
2405,Entre Rios,Caprino,Ovinocaprinocultura,205
2406,Entre Rios,Suíno total,Suinocultura,961
2407,Entre Rios,Lenha,Extrativismo,14


In [190]:
str_final = ''
for terr in terr_t1['pi_micro_m'].unique():

    str_final += f'\n### {terr} ###################################\n'
    
    terr_t1_filtrado = terr_t1[terr_t1['pi_micro_m'] == terr]
    
    for pot in terr_t1_filtrado['Potencialidade'].unique():
        _temp = terr_t1_filtrado[terr_t1_filtrado['Potencialidade'] == pot]
        _temp = _temp.groupby('subclasse').sum()[['Estoque_mun_t1']]
        _temp = _temp.sort_values('Estoque_mun_t1', ascending = False).iloc[0:5,]

        str_final += pot + ' - ' + ', '.join(_temp.index)+ '\n\n'
print(str_final)    


### Chapada Vale do Itaim ###################################
Indústria - Geração de energia elétrica, Fabricação de motores elétricos, peças e acessórios, Fabricação de artefatos de cerâmica e barro cozido para uso na construção, exceto azulejos e pisos, Coleta de resíduos não-perigosos, Fabricação de painéis e letreiros luminosos

Transporte, armazenagem e correio - Transporte rodoviário coletivo de passageiros, sob regime de fretamento, intermunicipal, interestadual e internacional, Transporte rodoviário de produtos perigosos

Construção - Construção de edifícios, Construção de estações e redes de telecomunicações

Atividades profissionais, científicas e técnicas - Cartórios, Atividades de contabilidade, Atividades de consultoria em gestão empresarial, exceto consultoria técnica específica, Serviços advocatícios, Serviços de engenharia

Polo de saúde - Serviços de diagnóstico por imagem com uso de radiação ionizante, exceto tomografia, Atividades de atendimento hospitalar, exceto p

In [191]:
with open('potencialidade_descrição_2025.txt', 'w', encoding="latin1") as file:
    file.write(str_final)

In [192]:
t1 = pd.read_excel('somente_t1_categorizado.xlsx', index_col = 0)
terr = pd.read_excel('C:/Users/matheus.barbosa/Desktop/Repositorios/Indicadores-Tabelas-Essenciais/Tabelas-Correlacao/territorios_desenvolvimento.xlsx')

terr_t1 = t1.merge(terr[['NM_MUN', 'pi_micro_m']], on = 'NM_MUN')

terr_t1 

,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade,pi_micro_m
0,"Captação, tratamento e distribuição de água",T1,Alto Longá,3,RAIS,pessoas,2022,Indústria,Entre Rios
1,Atividades de contabilidade,T1,Altos,5,RAIS,pessoas,2022,"Atividades profissionais, científicas e técnicas",Entre Rios
2,"Obras de urbanização - ruas, praças e calçadas",T1,Altos,16,RAIS,pessoas,2022,Construção,Entre Rios
3,Construção de rodovias e ferrovias,T1,Altos,8,RAIS,pessoas,2022,Construção,Entre Rios
4,Fabricação de produtos de padaria e confeitari...,T1,Altos,15,RAIS,pessoas,2022,Indústria,Entre Rios
...,...,...,...,...,...,...,...,...,...
2175,Melancia,T1,Água Branca,518,PAM,mil reais,2022,Hortifruticultura,Entre Rios
2176,Tomate,T1,Água Branca,700,PAM,mil reais,2022,Hortifruticultura,Entre Rios
2177,Ovos de galinha,T1,Água Branca,396,PPM,mil reais,2022,Avicultura,Entre Rios
2178,Ovino,T1,Água Branca,146,PPM,cabeças,2022,Ovinocaprinocultura,Entre Rios


In [193]:
terr_t1['Unidade de medida'].unique()[0]

'pessoas'

In [194]:
def trocar(x):
    valor = x
    if valor != 0:
        valor = 1
    return valor

for pot in terr_t1['Potencialidade'].values:

    tmp = terr_t1[terr_t1['Potencialidade'] == pot]

    fator = 1
    
    if tmp['Unidade de medida'].unique()[0] == 'mil reais':
        fator = 1000
    
    formatado = pd.pivot_table(tmp, index = 'NM_MUN', columns =  'subclasse', values = 'Estoque_mun_t1',
                   aggfunc = 'sum', fill_value=0)
    formatado = formatado*fator

    with pd.ExcelWriter(f'./Subclasses_dummy/Com_valor/{pot}_subclasses_com_valor_2023.xlsx', engine='openpyxl') as writer:  
        formatado.to_excel(writer)  
    
    for c in formatado.columns:
        formatado[c] = formatado[c].apply(trocar)

    with pd.ExcelWriter(f'./Subclasses_dummy/Binario/{pot}_subclasses_binario.xlsx', engine='openpyxl') as writer:  
        formatado.to_excel(writer)  

In [195]:
pd.read_excel('somente_t1_categorizado_2025.xlsx')['ANO'].value_counts().sum()

np.int64(2409)

In [196]:
pd.read_excel('somente_t1_categorizado.xlsx')['ANO'].value_counts().sum()

np.int64(2180)

In [197]:
t1 = pd.read_excel('somente_t1_categorizado_2025.xlsx')
t1

,Unnamed: 0,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade
0,0,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013,Indústria
1,1,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013,"Transporte, armazenagem e correio"
2,2,Provedores de acesso às redes de comunicações,T1,Agricolândia,7,RAIS,pessoas,2013,Informação e comunicação
3,3,Construção de edifícios,T1,Agricolândia,5,RAIS,pessoas,2013,Construção
4,4,Obras de alvenaria,T1,Alagoinha do Piauí,2,RAIS,pessoas,2013,Construção
...,...,...,...,...,...,...,...,...,...
2404,2404,Ovino,T1,Água Branca,148,PPM,cabeças,2013,Ovinocaprinocultura
2405,2405,Caprino,T1,Água Branca,205,PPM,cabeças,2013,Ovinocaprinocultura
2406,2406,Suíno total,T1,Água Branca,961,PPM,cabeças,2013,Suinocultura
2407,2407,Lenha,T1,Água Branca,14,PEVS,mil reais,2013,Extrativismo


In [198]:
t1[t1['Fonte'] == 'RAIS']

,Unnamed: 0,subclasse,classificacao_regiao,NM_MUN,Estoque_mun_t1,Fonte,Unidade de medida,ANO,Potencialidade
0,0,Fabricação de artefatos de cerâmica e barro co...,T1,Acauã,11,RAIS,pessoas,2013,Indústria
1,1,"Transporte rodoviário coletivo de passageiros,...",T1,Acauã,8,RAIS,pessoas,2013,"Transporte, armazenagem e correio"
2,2,Provedores de acesso às redes de comunicações,T1,Agricolândia,7,RAIS,pessoas,2013,Informação e comunicação
3,3,Construção de edifícios,T1,Agricolândia,5,RAIS,pessoas,2013,Construção
4,4,Obras de alvenaria,T1,Alagoinha do Piauí,2,RAIS,pessoas,2013,Construção
...,...,...,...,...,...,...,...,...,...
1173,1173,Laboratórios clínicos,T1,Água Branca,8,RAIS,pessoas,2013,Polo de saúde
1174,1174,Atividade médica ambulatorial restrita a consu...,T1,Água Branca,7,RAIS,pessoas,2013,Polo de saúde
1175,1175,Cartórios,T1,Água Branca,6,RAIS,pessoas,2013,"Atividades profissionais, científicas e técnicas"
1176,1176,"Lanchonetes, casas de chá, de sucos e similares",T1,Água Branca,19,RAIS,pessoas,2013,Alojamento e alimentação


In [199]:
for pot in t1['Potencialidade'].unique():

    print('\n')
    
    tmp  = t1[t1['Potencialidade'] == pot]
    subs = ', '.join(tmp['subclasse'].unique())

    print(pot + ' : ' + subs)



Indústria : Fabricação de artefatos de cerâmica e barro cozido para uso na construção, exceto azulejos e pisos, Fabricação de produtos de padaria e confeitaria com predominância de produção própria, Captação, tratamento e distribuição de água, Fabricação de alimentos para animais, Aparelhamento de pedras para construção, exceto associado à extração, Coleta de resíduos não-perigosos, Confecção, sob medida, de peças do vestuário, exceto roupas íntimas, Fabricação de outros produtos alimentícios não especificados anteriormente, Fabricação de farinha de mandioca e derivados, Moagem de trigo e fabricação de derivados, Fabricação de produtos de panificação industrial, Fabricação de móveis com predominância de metal, Fabricação de móveis com predominância de madeira, Serviços de usinagem, tornearia e solda, Manutenção e reparação de máquinas e equipamentos para agricultura e pecuária, Recuperação de materiais plásticos, Fabricação de produtos cerâmicos refratários, Aparelhamento de placas e

In [200]:
t1.pivot_table(index = 'Potencialidade', columns = 'ANO', aggfunc = 'count')['Fonte']#.sum()#.sum()

ANO,2013,2018,2022
Potencialidade,,,
,NaN,NaN,1.0
Agronegócio,229.0,66.0,188.0
Alojamento e alimentação,114.0,NaN,NaN
Apicultura e Meliponicultura,36.0,10.0,30.0
"Atividades profissionais, científicas e técnicas",122.0,NaN,NaN
Avicultura,53.0,12.0,114.0
Bovinocultura,12.0,7.0,22.0
Construção,161.0,NaN,NaN
Equinocultura,8.0,NaN,NaN


In [201]:
t1['Potencialidade'].value_counts().sum()

np.int64(2409)